# 1. Important environment configurations


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# Portfolio optimization libraries
from pypfopt import EfficientFrontier, expected_returns, risk_models
from pypfopt.discrete_allocation import DiscreteAllocation, get_latest_prices

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("✓ All libraries imported successfully")

# %%
# Configuration parameters
INITIAL_CAPITAL = 10000
SEQUENCE_LENGTH = 60  # Days to look back
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.2
LEARNING_RATE = 0.001
EPOCHS = 100
BATCH_SIZE = 32
RISK_FREE_RATE = 0.02  # 2% annual risk-free rate

# 20 stocks from 4 different industries
STOCKS = {
    'Technology': ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'META'],
    'Healthcare': ['JNJ', 'UNH', 'PFE', 'ABBV', 'LLY'],
    'Financial': ['JPM', 'BAC', 'WFC', 'GS', 'MS'],
    'Consumer': ['AMZN', 'TSLA', 'HD', 'MCD', 'NKE']
}

ALL_STOCKS = [stock for stocks in STOCKS.values() for stock in stocks]
BENCHMARK = '^GSPC'  # S&P 500

print(f"Configuration:")
print(f"  Initial Capital: ${INITIAL_CAPITAL:,}")
print(f"  Sequence Length: {SEQUENCE_LENGTH} days")
print(f"  Total Stocks: {len(ALL_STOCKS)}")
print(f"  LSTM Hidden Size: {HIDDEN_SIZE}")
print(f"  Training Epochs: {EPOCHS}")


# Model definition

In [ ]:
class LSTMPredictor(nn.Module):
    """
    LSTM-based stock price predictor
    
    Architecture:
    - Input: Sequence of historical prices for all stocks
    - LSTM layers: Process temporal patterns
    - Fully connected layers: Generate price prediction
    - Output: Predicted next-day price (scaled)
    """
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super(LSTMPredictor, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                           batch_first=True, dropout=dropout)
        self.fc1 = nn.Linear(hidden_size, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(64, 1)
        
    def forward(self, x):
        # x shape: (batch, seq_len, input_size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        out, _ = self.lstm(x, (h0, c0))
        out = out[:, -1, :]  # Take last time step
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out

# Test model instantiation
test_model = LSTMPredictor(input_size=20, hidden_size=HIDDEN_SIZE, 
                          num_layers=NUM_LAYERS, dropout=DROPOUT)
print(f"✓ LSTM Model created successfully")
print(f"  Total parameters: {sum(p.numel() for p in test_model.parameters()):,}")
